In [28]:
import re
from uuid import uuid4

from langchain_core.documents import Document
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma
from langchain_community.retrievers import BM25Retriever
from langchain_classic.retrievers import EnsembleRetriever

In [29]:
def tokenize(text: str) -> list[str]:
    return re.findall(r'\b\w+\b', text.lower())

def create_chunk(chunk_id: str, text: str) -> Document:
    return Document(page_content=text,metadata={"chunk_id": chunk_id})

In [30]:
scenarios = {
    # ---------------------------------------------------------
    # CAS 1 : le vectoriel devrait être meilleur
    # ---------------------------------------------------------
    "vector_better": {
        "query": (
            "How can a customer get their money back "
            "after buying an item?"
        ),
        "relevant_id": "vector_target",
        "expected_winner": "vector",
        "documents": [
            create_chunk(
                "vector_target",
                (
                    "A reimbursement may be requested during the "
                    "thirty days following a purchase. The amount "
                    "is returned to the original payment method."
                ),
            ),
            create_chunk(
                "vector_lexical_trap_1",
                (
                    "The customer account displays every item bought "
                    "and the total amount of money spent."
                ),
            ),
            create_chunk(
                "vector_lexical_trap_2",
                (
                    "Customers can compare item prices before buying "
                    "products from the online store."
                ),
            ),
            create_chunk(
                "vector_noise_1",
                (
                    "Users can change their payment method from "
                    "the account settings page."
                ),
            ),
            create_chunk(
                "vector_noise_2",
                (
                    "Order tracking shows when a purchased product "
                    "will arrive."
                ),
            ),
        ],
    },

    # ---------------------------------------------------------
    # CAS 2 : BM25 devrait être meilleur
    # ---------------------------------------------------------
    "bm25_better": {
        "query": "What is the retry limit for task AX-2047?",
        "relevant_id": "bm25_target",
        "expected_winner": "bm25",
        "documents": [
            create_chunk(
                "bm25_target",
                "AX-2047 | attempts_max=11 | queue=payments",
            ),
            create_chunk(
                "bm25_semantic_trap_1",
                (
                    "The retry limit for a failed payment-processing "
                    "task is three attempts."
                ),
            ),
            create_chunk(
                "bm25_semantic_trap_2",
                (
                    "Background tasks are executed again when a "
                    "temporary failure occurs."
                ),
            ),
            create_chunk(
                "bm25_wrong_code",
                (
                    "Task AX-2099 has a maximum retry limit of "
                    "seven attempts."
                ),
            ),
            create_chunk(
                "bm25_noise",
                (
                    "The queue worker stops processing a task after "
                    "its retry policy has been exhausted."
                ),
            ),
        ],
    },

    # ---------------------------------------------------------
    # CAS 3 : l'hybride devrait être meilleur
    # ---------------------------------------------------------
   "hybrid_better": {
    "query": "ZX-41",
    "relevant_id": "hybrid_target",
    "expected_winner": "hybrid",
    "documents": [
        create_chunk(
            "hybrid_target",
            (
                "ZX-41 stores archived customer records "
                "in an encrypted backup vault used by ZX-41, this sentence is to make"
                "ZX-41 the most relevant document for the query ZX-41."
            ),
        ),
        create_chunk(
            "hybrid_lexical_trap",
            (
                "ZX-41 validates customer identifiers "
                "and generates audit reports."
            ),
        ),
        create_chunk(
            "hybrid_semantic_trap",
            (
                "Old client information is preserved "
                "in secure secondary storage."
            ),
        ),
        create_chunk(
            "hybrid_semantic_noise_1",
            (
                "Archived records are deleted after "
                "the legal retention period."
            ),
        ),
        create_chunk(
            "hybrid_code_noise",
            (
                "ZX-41 runs every night at 2 AM."
            ),
        ),
        create_chunk(
            "hybrid_noise_1",
            (
                "Customer databases contain names, "
                "addresses and payment details."
            ),
        ),
        create_chunk(
            "hybrid_noise_2",
            (
                "The backup system restores information "
                "after server failures."
            ),
        ),
        create_chunk(
            "hybrid_noise_3",
            (
                "Encrypted vaults protect sensitive "
                "financial documents."
            ),
        ),
    ],
},
}

In [31]:
from llms import get_embedding_model

embedding_model = get_embedding_model()

def create_retrievers(documents: list[Document], scenario_name: str):
    ids = [document.metadata["chunk_id"] for document in documents]
    
    collection_name = "benchmark"+scenario_name
    
    db = Chroma.from_documents(
        documents=documents,
        embedding=embedding_model,
        ids=ids,
        collection_name=collection_name,
        collection_metadata={"hnsw:space":"cosine"}
    )
    
    vector_retriever = db.as_retriever(search_type="similarity",
                                       search_kwargs={"k": 3})
    
    bm25_retriever = BM25Retriever.from_documents(documents=documents, preprocess_func=tokenize)
    
    hybrid_retriever = EnsembleRetriever(retrievers=[vector_retriever, bm25_retriever], weights=[0.4,0.6])
    
    return {"vector": vector_retriever, "bm25": bm25_retriever, "hybrid": hybrid_retriever}

In [32]:
def get_rank(results: list[Document], relevant_id: str) -> int | None:
    for rank, document in enumerate(results, start=1): 
        if document.metadata["chunk_id"] == relevant_id:
            return rank
    return None
    

In [33]:
def print_ranking(
    retriever_name: str,
    results: list[Document],
    relevant_id: str,
) -> None:
    print(f"\n--- {retriever_name.upper()} ---")

    for rank, document in enumerate(results, start=1):
        chunk_id = document.metadata["chunk_id"]

        marker = (
            "  <-- CHUNK PERTINENT"
            if chunk_id == relevant_id
            else ""
        )

        print(f"{rank}. {chunk_id}{marker}")
        print(f"   {document.page_content}")

In [34]:
benchmark_results = []

for scenario_name, scenario in scenarios.items():
    print("\n" + "=" * 100)
    print(f"SCÉNARIO : {scenario_name}")
    print(f"Question : {scenario['query']}")
    print(f"Chunk attendu : {scenario['relevant_id']}")
    print(f"Retriever attendu : {scenario['expected_winner']}")

    retrievers = create_retrievers(
        scenario["documents"],
        scenario_name
    )

    for retriever_name, retriever in retrievers.items():
        results = retriever.invoke(
            scenario["query"]
        )

        rank = get_rank(
            results,
            scenario["relevant_id"],
        )

        benchmark_results.append({
            "scenario": scenario_name,
            "retriever": retriever_name,
            "rank": rank,
        })

        print_ranking(
            retriever_name,
            results,
            scenario["relevant_id"],
        )


SCÉNARIO : vector_better
Question : How can a customer get their money back after buying an item?
Chunk attendu : vector_target
Retriever attendu : vector

--- VECTOR ---
1. vector_target  <-- CHUNK PERTINENT
   A reimbursement may be requested during the thirty days following a purchase. The amount is returned to the original payment method.
2. vector_lexical_trap_1
   The customer account displays every item bought and the total amount of money spent.
3. vector_noise_1
   Users can change their payment method from the account settings page.

--- BM25 ---
1. vector_lexical_trap_1
   The customer account displays every item bought and the total amount of money spent.
2. vector_lexical_trap_2
   Customers can compare item prices before buying products from the online store.
3. vector_noise_1
   Users can change their payment method from the account settings page.
4. vector_target  <-- CHUNK PERTINENT
   A reimbursement may be requested during the thirty days following a purchase. The a

In [35]:
def calculate_metrics(
    rows: list[dict],
    retriever_name: str,
) -> dict:
    retriever_rows = [
        row
        for row in rows
        if row["retriever"] == retriever_name
    ]

    total = len(retriever_rows)

    hit_at_1 = sum(
        row["rank"] == 1
        for row in retriever_rows
    ) / total

    hit_at_3 = sum(
        row["rank"] is not None
        and row["rank"] <= 3
        for row in retriever_rows
    ) / total

    mrr = sum(
        1 / row["rank"]
        if row["rank"] is not None
        else 0
        for row in retriever_rows
    ) / total

    return {
        "hit@1": hit_at_1,
        "hit@3": hit_at_3,
        "mrr": mrr,
    }


print("\n" + "=" * 100)
print("RÉSULTATS GLOBAUX")

for retriever_name in ["vector", "bm25", "hybrid"]:
    metrics = calculate_metrics(
        benchmark_results,
        retriever_name,
    )

    print(f"\n{retriever_name.upper()}")
    print(f"Hit at 1 : {metrics['hit@1']:.3f}")
    print(f"Hit at 3 : {metrics['hit@3']:.3f}")
    print(f"MRR   : {metrics['mrr']:.3f}")


RÉSULTATS GLOBAUX

VECTOR
Hit at 1 : 0.333
Hit at 3 : 0.667
MRR   : 0.444

BM25
Hit at 1 : 0.333
Hit at 3 : 0.667
MRR   : 0.583

HYBRID
Hit at 1 : 0.333
Hit at 3 : 0.667
MRR   : 0.583


In [36]:
print("\n" + "=" * 100)
print("RANG DU BON CHUNK")

for scenario_name in scenarios:
    print(f"\n{scenario_name}")

    for row in benchmark_results:
        if row["scenario"] == scenario_name:
            print(
                f"{row['retriever']:>8} : "
                f"rang {row['rank']}"
            )


RANG DU BON CHUNK

vector_better
  vector : rang 1
    bm25 : rang 4
  hybrid : rang 2

bm25_better
  vector : rang None
    bm25 : rang 2
  hybrid : rang 4

hybrid_better
  vector : rang 3
    bm25 : rang 1
  hybrid : rang 1


As shown by the results, hybrid search can be useful for retrieving chunks that contain specific keywords or identifiers that vector search alone may overlook.
